# Knowledge Graph Construction

In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached https://download.pytorch.org/whl/cpu/torch-2.9.1%2Bcpu-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached https://download.pytorch.org/whl/cpu/torchvision-0.24.1%2Bcpu-cp313-cp313-win_amd64.whl.metadata (6.1 kB)
  Using cached https://download.pytorch.org/whl/cpu/torchaudio-2.9.1%2Bcpu-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
Using cached https://download.pytorch.org/whl/cpu/torch-2.9.1%2Bcpu-cp313-cp313-win_amd64.whl (110.9 MB)
Using cached https://download.pytorch.org/whl/cpu/torchvision-0.24.1%2Bcpu-cp313-cp313-win_amd64.whl (4.3 MB)
Using cached https://download.pytorch.org/whl/cpu/torchaudio-2.9.1%2Bcpu-cp313-cp313-win_amd64.whl (663 kB)

   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   -----------------

In [9]:
%pip install sentence-transformers networkx openai numpy ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [22]:
%pip install ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
EMBED_MODEL = "all-MiniLM-L6-v2"     
LLM_MODEL = "qwen2.5:7b"            

In [ ]:
import json, os, re, numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer

In [ ]:
try:
    from ollama import Client
    ollama_client = Client(host='http://localhost:11434')
    print("Using Ollama Client")
except ImportError:
    # Fallback to direct HTTP requests
    import requests
    ollama_client = None
    print("Using direct HTTP requests for Ollama")


SCRAPED_PATH = "scraped_content.txt"
VECTOR_DB_PATH = "chroma_db"
KG_JSON_PATH = "kg.json"

Using Ollama Client


In [ ]:
from ollama import chat
import ollama
ollama.pull("qwen2.5:7b")
if ollama_client:
    response = ollama_client.chat(
        model="qwen2.5:7b",  
        messages=[
            {"role": "user", "content": "Hello world!"}
        ],
        stream = False
    )
    print("Model response:")
    print(response.message.content[:])
else:
    print("Ollama client not available, cannot run chat.")

Model response:
Hello! How can I assist you today?


Test KG Creation

In [121]:
from ollama import Client, pull

SCRAPED_PATH = "scraped_content.txt"
EMBED_MODEL = "all-MiniLM-L6-v2"   # Local embedding model
LLM_MODEL = "qwen2.5:7b"           # Ollama local model
KG_FILE = "dummy_knowledge_graph.gpickle"
embedder = SentenceTransformer(EMBED_MODEL)

In [116]:
with open(SCRAPED_PATH, "r", encoding="utf-8") as f:
    full_text = f.read()

print("Loaded text:", len(full_text), "characters")

Loaded text: 12474141 characters


In [117]:
def extract_concepts(text):
    prompt = f"""
Extract ALL AI/ML concepts mentioned in this text.

For each concept include:
- name
- short definition (from context only)
- aliases / synonyms
- difficulty (easy/medium/hard)

Text:
{text[:20000]}  # limit for long text

Return JSON list with format:
[
  {{
    "name": "...",
    "definition": "...",
    "aliases": ["...", ...],
    "difficulty": "medium"
  }}
]
"""
    response = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=False
    )
    data = json.loads(response.message.content)
    print("HERE I AM:", data)
    return data #["concepts"]

concepts = extract_concepts(full_text)
print("Extracted concepts:", len(concepts))


HERE I AM: [{'name': 'Introduction to MDP (Markov Decision Processes)', 'definition': 'A framework for modeling decision making in situations where outcomes are partly random and partly under the control of a decision maker. It consists of states, actions, rewards, transition probabilities, and policies.', 'aliases': ['MDP', 'Markov Decision Process'], 'difficulty': 'medium'}, {'name': 'Policy Iteration in MDP', 'definition': 'A method for finding the optimal policy by alternately improving the policy evaluation step (value iteration) and then using this value function to improve the policy.', 'aliases': ['Policy Improvement', 'Value Iteration'], 'difficulty': 'medium'}, {'name': 'Temporal Difference (TD) Learning', 'definition': 'A method for predicting state values or action-values by combining bootstrapping, which uses current estimates to improve future estimates, with eligibility traces, a mechanism that allows the updating of multiple states in one step.', 'aliases': ['TD(0)', 'S

In [118]:
# CREATE EMBEDDINGS FOR THE CONCEPTS
names = [c["name"] for c in concepts]
embeddings = embedder.encode(names, normalize_embeddings=True)


In [119]:
# BUILD KNOWLEDGE GRAPH
G = nx.DiGraph()

for concept, emb in zip(concepts, embeddings):
    G.add_node(
        concept["name"],
        type="concept",
        definition=concept["definition"],
        difficulty=concept["difficulty"],
        aliases=concept["aliases"],
        embedding=emb.astype(float).tolist()
    )

In [120]:
def cosine(a, b):
    return np.dot(a, b)

threshold = 0.55
for i, c1 in enumerate(concepts):
    for j, c2 in enumerate(concepts):
        if i >= j:
            continue
        sim = cosine(embeddings[i], embeddings[j])
        if sim >= threshold:
            G.add_edge(c1["name"], c2["name"], relation="related_to", weight=float(sim))
            G.add_edge(c2["name"], c1["name"], relation="related_to", weight=float(sim))

In [122]:
def infer_prereqs(concepts):
    concept_list = ", ".join([c["name"] for c in concepts])
    prompt = f"""
Here are some AI/ML concepts:

{concept_list}

Infer prerequisite relationships between concepts.
Use your understanding of AI—but ONLY return edges that are logically valid.

Return JSON:
{{
   "prerequisites": [
     {{"from": "probability", "to": "bayes rule"}},
     ...
   ]
}}
"""
    response = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=False
    )

    print("HERE I AM1:",type(response),response)
    #return json.loads(response.message.content)["prerequisites"]
    return response


In [123]:
def parse_json_from_ollama(text):
    """
    Extract JSON content from Ollama response text.
    """
    # Search for a JSON code block first
    match = re.search(r"```json(.*?)```", text, re.DOTALL)
    if match:
        json_text = match.group(1).strip()
        return json.loads(json_text)
    
    # Fallback: try to parse the whole string
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("Warning: Could not parse JSON")
        return None

# Example usage:
raw_text = infer_prereqs(concepts).message.content
data = parse_json_from_ollama(raw_text)
prereq_edges = data.get("prerequisites", []) if data else []
print("Parsed prerequisites:", prereq_edges)

HERE I AM1: <class 'ollama._types.ChatResponse'> model='qwen2.5:7b' created_at='2025-12-06T00:58:42.3051245Z' done=True done_reason='stop' total_duration=4580873200 load_duration=69459500 prompt_eval_count=176 prompt_eval_duration=408906100 eval_count=204 eval_duration=3172803300 message=Message(role='assistant', content='```json\n{\n   "prerequisites": [\n     {"from": "MDP (Markov Decision Processes)", "to": "Probability Theory"},\n     {"from": "Policy Iteration in MDP", "to": "MDP (Markov Decision Processes)"},\n     {"from": "Temporal Difference (TD) Learning", "to": "Reinforcement Learning"},\n     {"from": "Reinforcement Learning", "to": "Markov Decision Processes (MDP)"},\n     {"from": "Maximum Likelihood Estimation (MLE)", "to": "Probability Theory"},\n     {"from": "Convolutional Neural Networks (CNNs)", "to": "Backpropagation"},\n     {"from": "Feature Extraction via Residual Networks (ResNet)", "to": "Convolutional Neural Networks (CNNs)"},\n     {"from": "Scene Understand

In [124]:
#prereq_edges = infer_prereqs(concepts)
for edge in prereq_edges:
    src = edge["from"]
    dst = edge["to"]
    if src in G.nodes() and dst in G.nodes():
        G.add_edge(src, dst, relation="prereq_of")


In [125]:
import pickle

with open(KG_FILE, "wb") as f:
    pickle.dump(G, f)
print(f"Graph saved → {KG_FILE}")
print("Nodes:", len(G.nodes()))
print("Edges:", len(G.edges()))


Graph saved → dummy_knowledge_graph.gpickle
Nodes: 14
Edges: 7


# Building the Knowledge Graph on the course website content

In [ ]:
import os, json, re, numpy as np, pickle
import networkx as nx
from sentence_transformers import SentenceTransformer
from ollama import Client, pull


In [ ]:
SCRAPED_PATH = "scraped_content.txt"
EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "qwen2.5:7b"
KG_FILE = "knowledge_graph.pkl"  

CHUNK_SIZE = 60000 #20000
CHUNK_OVERLAP = 3000 #1000
EMBED_NORMALIZE = True
REL_SIM_THRESHOLD = 0.40


In [ ]:
ollama_client = Client(host="http://localhost:11434")  
embedder = SentenceTransformer(EMBED_MODEL)


In [ ]:
with open(SCRAPED_PATH, "r", encoding="utf-8") as f:
    full_text = f.read()
print("Loaded text:", len(full_text), "characters")

Loaded text: 12474141 characters


In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


In [ ]:
def parse_json_from_ollama(text):

    results = []

    # 1. Extract ```json ... ``` blocks if present
    matches = re.findall(r"```json(.*?)```", text, re.DOTALL | re.IGNORECASE)
    chunks_to_parse = matches if matches else [text]

    for chunk in chunks_to_parse:
        # sanitize
        chunk = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', '', chunk)
        chunk = chunk.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")

        # Attempt full JSON parse first
        try:
            parsed = json.loads(chunk)
            if isinstance(parsed, list):
                results.extend(parsed)
            else:
                results.append(parsed)
            continue
        except json.JSONDecodeError:
            pass

        # fallback: parse multiple JSON objects in sequence
        for obj_str in re.findall(r'\{.*?\}', chunk, re.DOTALL):
            try:
                results.append(json.loads(obj_str))
            except json.JSONDecodeError:
                continue

    return results



In [ ]:
def extract_concepts(chunk):
    prompt = f"""
Extract ALL AI/ML concepts mentioned in this text.

For each concept include:
- name
- short definition (from context only)
- aliases / synonyms
- difficulty (easy/medium/hard)

Text:
{chunk}

Return JSON list with format:
[
  {{
    "name": "...",
    "definition": "...",
    "aliases": ["...", ...],
    "difficulty": "medium"
  }}
]
"""
    response = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=False
    )
    return parse_json_from_ollama(response.message.content)


In [ ]:
chunks = chunk_text(full_text)
all_concepts = []

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i+1}/{len(chunks)}")
    concepts_chunk = extract_concepts(chunk)
    if concepts_chunk:
        # flatten the list in case multiple JSON objects returned
        if isinstance(concepts_chunk, list):
            all_concepts.extend(concepts_chunk)
        else:
            all_concepts.append(concepts_chunk)


# Remove duplicates
unique_concepts = {c["name"]: c for c in all_concepts}
concepts = list(unique_concepts.values())
print("Total concepts extracted:", len(concepts))



Processing chunk 1/219
Processing chunk 2/219
Processing chunk 3/219
Processing chunk 4/219
Processing chunk 5/219
Processing chunk 6/219
Processing chunk 7/219
Processing chunk 8/219
Processing chunk 9/219
Processing chunk 10/219
Processing chunk 11/219
Processing chunk 12/219
Processing chunk 13/219
Processing chunk 14/219
Processing chunk 15/219
Processing chunk 16/219
Processing chunk 17/219
Processing chunk 18/219
Processing chunk 19/219
Processing chunk 20/219
Processing chunk 21/219
Processing chunk 22/219
Processing chunk 23/219
Processing chunk 24/219
Processing chunk 25/219
Processing chunk 26/219
Processing chunk 27/219
Processing chunk 28/219
Processing chunk 29/219
Processing chunk 30/219
Processing chunk 31/219
Processing chunk 32/219
Processing chunk 33/219
Processing chunk 34/219
Processing chunk 35/219
Processing chunk 36/219
Processing chunk 37/219
Processing chunk 38/219
Processing chunk 39/219
Processing chunk 40/219
Processing chunk 41/219
Processing chunk 42/219
P

In [98]:
concepts

[{'name': 'Camera Calibration and Distortion Correction Process',
  'definition': 'The process of determining the intrinsic parameters (camera matrix) and distortion coefficients for a camera, allowing for the removal of lens distortions from captured images.',
  'aliases': ['calibration', 'camera matrix', 'distortion correction'],
  'difficulty': 'medium'},
 {'name': 'Reprojection Error Analysis',
  'definition': 'The evaluation of how accurately projected 3D points match their corresponding 2D image points after undistorting the images, providing a measure of calibration quality.',
  'aliases': ['reprojection error', 'undistortion accuracy'],
  'difficulty': 'medium'},
 {'name': 'Intrinsic Camera Parameters',
  'definition': 'Properties of a camera that are related to its internal characteristics and do not depend on the scene being imaged, including focal lengths (fx, fy) and principal point (cx, cy).',
  'aliases': ['camera matrix', 'intrinsic parameters'],
  'difficulty': 'medium'

In [99]:
concept_names = []
for i in concepts:
    concept_names.append(i['name'])

concept_names



['Camera Calibration and Distortion Correction Process',
 'Reprojection Error Analysis',
 'Intrinsic Camera Parameters',
 'Lens Distortion Models',
 'xref',
 'obj',
 'stream',
 'endstream',
 'endobj',
 'Extract Information from Text',
 'Extract Text',
 'Extract Text Between Tags',
 'Extract Text Information',
 'Extract JSON data from text',
 'Extract Text and Information',
 'Pattern1',
 'Extracted Data',
 'Aldorin',
 'Example Entity',
 'Extract JSON from Text',
 'Extract JSON Data',
 'Extract JSON information from text',
 'LOD',
 'KÏ',
 'V&:%`',
 'Y�S',
 'JSON Format',
 'JSON List Conversion',
 'Extract Content',
 'Extract JSON list from text',
 'extractText',
 'BitsPerComponent',
 'ColorSpace',
 'Filter',
 'Height',
 'Interpolate',
 'Length',
 'Subtype',
 'Type',
 'Width',
 'J',
 'M',
 'K',
 'Extract Text Pattern',
 'QEQE',
 'Extract Text from PDF',
 'Identify Image Types and Content',
 'L',
 'Extract JSON list',
 'Extract JSON',
 'W',
 'Ez',
 'Extracting Information',
 'JSON List',
 

In [ ]:
import re

def is_valid_concept_name(name: str) -> bool:
    name = name.strip()

    if not name:
        return False
    
    # Remove corrupted unicode / gibberish (allow some East Asian chars)
    if re.search(r'[^\x00-\x7F]', name) and not re.search(r'[가-힣一-龥ぁ-んァ-ン]', name):
        return False

    # Remove too-short tokens
    if len(name) <= 3 and "AI" not in name:
        return False

    # Remove PDF operators
    pdf_ops = ["xref", "obj", "endobj", "stream", "endstream",
               "BitsPerComponent", "ColorSpace", "Subtype", "Filter"]
    if name in pdf_ops:
        return False

    # Remove font names / font-like tokens
    if re.search(r'(CM|LM|MathItalic|Regular|Bold|Font|CMBX)\d', name):
        return False
    if "Font" in name or "Regular" in name or "Bold" in name:
        return False

    # Remove programming tokens (all-lowercase single words)
    if re.match(r'^[a-z_]+$', name):
        return False

    # Remove random names / sample data
    blacklist = ["Gandalf", "Harry Potter", "Aldorin", "Example Name", "Zafer"]
    if name in blacklist:
        return False

    # Remove numeric-only or symbol-only
    if re.match(r'^[\d\W]+$', name):
        return False

    # Remove assignment/admin text
    admin_words = ["Assignment", "Repository", "Submission", "Document", "Preview"]
    if any(w in name for w in admin_words):
        return False

    # Remove patterns like subsubsection.3.6.4, equation.2.1
    if re.match(r'^[A-Za-z]+\.\d+(\.\d+)*$', name):
        return False

    return True

# Clean the concepts list
cleaned_concepts = [c for c in concepts if is_valid_concept_name(c['name'])]

print(f"Kept {len(cleaned_concepts)}/{len(concepts)} concepts")
print([c['name'] for c in cleaned_concepts[:20]]) 


Kept 207/295 concepts
['Camera Calibration and Distortion Correction Process', 'Reprojection Error Analysis', 'Intrinsic Camera Parameters', 'Lens Distortion Models', 'Extract Information from Text', 'Extract Text', 'Extract Text Between Tags', 'Extract Text Information', 'Extract JSON data from text', 'Extract Text and Information', 'Pattern1', 'Extracted Data', 'Example Entity', 'Extract JSON from Text', 'Extract JSON Data', 'Extract JSON information from text', 'V&:%`', 'JSON Format', 'JSON List Conversion', 'Extract Content']


In [101]:
cleaned_concepts

[{'name': 'Camera Calibration and Distortion Correction Process',
  'definition': 'The process of determining the intrinsic parameters (camera matrix) and distortion coefficients for a camera, allowing for the removal of lens distortions from captured images.',
  'aliases': ['calibration', 'camera matrix', 'distortion correction'],
  'difficulty': 'medium'},
 {'name': 'Reprojection Error Analysis',
  'definition': 'The evaluation of how accurately projected 3D points match their corresponding 2D image points after undistorting the images, providing a measure of calibration quality.',
  'aliases': ['reprojection error', 'undistortion accuracy'],
  'difficulty': 'medium'},
 {'name': 'Intrinsic Camera Parameters',
  'definition': 'Properties of a camera that are related to its internal characteristics and do not depend on the scene being imaged, including focal lengths (fx, fy) and principal point (cx, cy).',
  'aliases': ['camera matrix', 'intrinsic parameters'],
  'difficulty': 'medium'

In [ ]:
names = [c["name"] for c in cleaned_concepts]
embeddings = embedder.encode(names, normalize_embeddings=EMBED_NORMALIZE)

In [ ]:
G = nx.DiGraph()
for concept, emb in zip(cleaned_concepts, embeddings):
    '''G.add_node(
        concept["name"],
        type="concept",
        definition=concept["definition"],
        difficulty=concept["difficulty"],
        aliases=concept["aliases"],
        embedding=emb.astype(float).tolist()
    )'''
    #for concept, emb in zip(concepts, embeddings):
    G.add_node(
        concept.get("name", "").strip(),
        type="concept",
        definition=concept.get("definition", "").strip(),
        difficulty=concept.get("difficulty", "unknown"),
        aliases=concept.get("aliases", []),
        embedding=emb.astype(float).tolist()
    )


In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)
    #return np.dot(a, b)

for i, c1 in enumerate(cleaned_concepts):
    for j, c2 in enumerate(cleaned_concepts):
        if i >= j:
            continue
        sim = cosine(embeddings[i], embeddings[j])
        if sim >= REL_SIM_THRESHOLD:
            G.add_edge(c1["name"], c2["name"], relation="related_to", weight=float(sim))
            G.add_edge(c2["name"], c1["name"], relation="related_to", weight=float(sim))


In [ ]:
def infer_prereqs(cleaned_concepts):
    concept_list = ", ".join([c["name"] for c in cleaned_concepts])
    prompt = f"""
    Here are some AI/ML concepts:

    {concept_list}

    Infer prerequisite relationships between concepts.
    Return JSON with format:

    ```json
    {{
    "prerequisites": [
        {{"from": "concept A", "to": "concept B"}}
    ]
    }}
    """
    response = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=False
    )
    data = parse_json_from_ollama(response.message.content)
    print("HERE:", data)

    if isinstance(data, dict):
        return data.get("prerequisites", [])

    elif isinstance(data, list):
        return data

    return []


prereq_edges = infer_prereqs(cleaned_concepts)

for edge in prereq_edges:

    # skip totally invalid formats
    if not isinstance(edge, dict):
        continue

    src = edge.get("from")
    dst = edge.get("to")

    # only source exists
    if src and not dst:
        if src not in G.nodes():
            G.add_node(src, type="concept", definition="", difficulty="unknown",
                       aliases=[], embedding=[])
        continue

    # only destination exists
    if dst and not src:
        if dst not in G.nodes():
            G.add_node(dst, type="concept", definition="", difficulty="unknown",
                       aliases=[], embedding=[])
        continue

    # both exist, add edge
    if src and dst:
        if src not in G.nodes():
            G.add_node(src, type="concept", definition="", difficulty="unknown",
                       aliases=[], embedding=[])
        if dst not in G.nodes():
            G.add_node(dst, type="concept", definition="", difficulty="unknown",
                       aliases=[], embedding=[])
        G.add_edge(src, dst, relation="prereq_of")


HERE: [{'prerequisites': [{'from': 'Camera Calibration and Distortion Correction Process', 'to': 'Intrinsic Camera Parameters'}, {'from': 'Camera Calibration and Distortion Correction Process', 'to': 'Lens Distortion Models'}, {'from': 'Reprojection Error Analysis', 'to': 'Camera Calibration'}, {'from': 'Extract Information from Text', 'to': 'Extract Text'}, {'from': 'Extract Information from Text', 'to': 'Extract Text Between Tags'}, {'from': 'Extract JSON data from text', 'to': 'Extract Text and Information'}, {'from': 'Extract JSON information from text', 'to': 'Extract JSON data from a given text.'}, {'from': 'Pattern1', 'to': 'Extracted Data'}, {'from': 'Example Entity', 'to': 'Extracted Data'}, {'from': 'JSON Format', 'to': 'Extract JSON data'}, {'from': 'Format Transformation', 'to': 'Transforming Equations'}, {'from': 'Definition of Keys and Sections', 'to': 'Sample Entry'}, {'from': 'Extracting Annotations and References', 'to': 'Example Extraction'}, {'from': 'Training Latent

In [ ]:
#--------------------------
#7. SAVE GRAPH USING PICKLE
#--------------------------

#with open(KG_FILE, "wb") as f:
with open("knowledge_graph1.pkl", "wb") as f:
    
    pickle.dump(G, f)

print(f"Graph saved → {"knowledge_graph1.pkl"}")
print("Nodes:", len(G.nodes()))
print("Edges:", len(G.edges()))


Graph saved at knowledge_graph1.pkl
Nodes: 14
Edges: 7


In [62]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   --------------- ------------------------ 4.2/11.0 MB 28.9 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 35.3 MB/s  0:00:00

   ---------------------------------------- 0/2 [pytz]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   ------------

In [107]:
import pandas as pd

df_nodes = pd.DataFrame([
    {
        "concept": n,
        "type": d.get("type"),
        "difficulty": d.get("difficulty"),
        "has_embedding": bool(d.get("embedding")),
        "aliases": ", ".join(d.get("aliases", [])),
        "definition": (d.get("definition")[:120] + "...") if d.get("definition") else ""
    }
    for n, d in G.nodes(data=True)
])

df_nodes.head(100)


,concept,type,difficulty,has_embedding,aliases,definition
0,Camera Calibration and Distortion Correction P...,concept,medium,True,"calibration, camera matrix, distortion correction",The process of determining the intrinsic param...
1,Reprojection Error Analysis,concept,medium,True,"reprojection error, undistortion accuracy",The evaluation of how accurately projected 3D ...
2,Intrinsic Camera Parameters,concept,medium,True,"camera matrix, intrinsic parameters",Properties of a camera that are related to its...
3,Lens Distortion Models,concept,medium,True,"distortion coefficients, lens correction",Mathematical models used to correct for radial...
4,Extract Information from Text,concept,medium,True,"Information Extraction, Text Mining",The task involves identifying and extracting s...
...,...,...,...,...,...,...
95,Loss Value Tracking,concept,medium,True,"Training Loss, Model Performance Tracking",The provided data represents a sequence of los...
96,Extract Loss Values from Training Data,concept,medium,True,"Loss Extraction, Training Loss Data",The task involves extracting the loss values f...
97,Loss Value Tracking in Training Loop,concept,medium,True,"Training Loss History, Epoch-wise Loss Values",The provided JSON is a sequence of loss values...
98,Extracting Information from Text,concept,medium,True,"Data Extraction, Information Retrieval",The process of identifying and extracting spec...
